### Topic modeling

In [2]:
!pip install bertopic
!pip install sentence-transformers

In [46]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd
import re
from collections import Counter

# modelos
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer

# componentes de BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer
from bertopic.representation import KeyBERTInspired, MaximalMarginalRelevance

In [4]:
df_pelis = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/data/peliculas_limpio.csv")

In [5]:
usuarios = pd.read_csv("https://raw.githubusercontent.com/nazarenomm/Sistema-de-recomendacion-de-peliculas/refs/heads/main/data/usuarios.csv")

## Preprocesado

In [6]:
def limpiar_texto(text):
    if not isinstance(text, str):
        return ''
    text = re.sub(r'<[^>]+>', ' ', text)        # remover HTML si hubiera
    text = re.sub(r'[^\w\s\.,;:!?áéíóúüñ-]', ' ', text)  # caracteres extraños
    text = re.sub(r'\s+', ' ', text)             # espacios múltiples
    return text.strip()

In [7]:
df_pelis["texto"] = (
    df_pelis["name"].apply(limpiar_texto) + ". "
    + df_pelis["description"].apply(limpiar_texto) + " "
    + df_pelis["director"].fillna('').apply(limpiar_texto) + ". "
    + df_pelis["year"].apply(lambda x: str(int(x)) if pd.notnull(x) else '') + ". "
    + df_pelis["genre"].apply(limpiar_texto) + ". "
    + df_pelis["keywords"].apply(limpiar_texto)
)

In [8]:
df_pelis.texto.iloc[0]

'Herida abierta. Orin Boyd, un duro policía de una comisaría del centro de la ciudad, descubre una red de policías corruptos. Andrzej Bartkowiak. 2001. acción, crimen, suspense. vietnam war veteran, heroína, drogas, narcotraficante, corrupt cop'

## Entrenamiento

In [9]:
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [10]:
pelis_embeddings = model.encode(df_pelis["texto"].tolist(), show_progress_bar=True)

Batches:   0%|          | 0/156 [00:00<?, ?it/s]

## Enfoque 3 — Topic Modeling con BERTopic

Tercera estrategia de representación, conceptualmente distinta a las anteriores. En vez de comparar embeddings densos directamente, **descubrimos tópicos latentes** en el corpus con BERTopic (reducción de dimensionalidad + clustering sobre los embeddings) y representamos cada película por su **distribución de probabilidad sobre esos tópicos**.

A cada usuario lo representamos del mismo modo (su texto query+historial proyectado a la distribución de tópicos) y recomendamos por similitud entre distribuciones.

### Instanciamos los componentes de BERTopic

Algoritmo de reducción dimensional

In [50]:
umap_model = UMAP(
    n_neighbors=10,    # más alto = estructura global, más bajo = local
    n_components=5,    # dimensiones de salida antes del clustering
    min_dist=0.0,      # 0.0 produce clusters más compactos
    metric='cosine'    # consistente con embeddings
)

Clustering

In [ ]:
hdbscan_model = HDBSCAN(
    min_cluster_size=5,   # mínimo de docs por tópico
    min_samples=5,         # controla qué tan conservador es con outliers
    metric='euclidean',    # sobre el espacio reducido por UMAP
    cluster_selection_method='eom',  # o 'leaf' para clusters más pequeños
    prediction_data=True
)

Tokenización/vectorización: Afecta cómo se representan los tópicos (las palabras clave), no el clustering en sí.

In [13]:
vectorizer_model = CountVectorizer(
    ngram_range=(1, 2),     # incluir bigramas en la representación
    stop_words=None,         # lista con stopwords personalizada o None para usar la predeterminada
    min_df=2                 # ignorar términos muy raros
)

Representación de tópicos

In [14]:
representation_model = KeyBERTInspired()

### Instanciamos BERTopic

Instanciamos con configuración mínima, vamos a usar BERTopic con sus componentes por defecto.

Le pasamos los embeddings precomputados y el mismo _transformer_ que venimos usando.

In [59]:
topic_model = BERTopic(
    embedding_model=model,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    # vectorizer_model=vectorizer_model,
    representation_model=representation_model,
    calculate_probabilities=True,
    nr_topics="auto",
    verbose=True
)

topics, probs = topic_model.fit_transform(df_pelis["texto"].tolist(), pelis_embeddings)

2026-06-18 23:17:09,374 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-06-18 23:17:16,226 - BERTopic - Dimensionality - Completed ✓
2026-06-18 23:17:16,228 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-06-18 23:17:25,174 - BERTopic - Cluster - Completed ✓
2026-06-18 23:17:25,176 - BERTopic - Representation - Extracting topics using c-TF-IDF for topic reduction.
2026-06-18 23:17:25,609 - BERTopic - Representation - Completed ✓
2026-06-18 23:17:25,610 - BERTopic - Topic reduction - Reducing number of topics
2026-06-18 23:17:25,691 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-06-18 23:18:30,128 - BERTopic - Representation - Completed ✓
2026-06-18 23:18:30,133 - BERTopic - Topic reduction - Reduced number of topics from 189 to 74


Vemos la cantidad de tópicos encontrados y su count

In [60]:
# Número de tópicos encontrados (excluye -1 = outliers)
n_topics = len(topic_model.get_topics()) - 1
print(f"Tópicos encontrados: {n_topics}")
print(f"Outliers (tópico -1): {topics.count(-1)}")

Tópicos encontrados: 73
Outliers (tópico -1): 2155


In [61]:
topic_model.get_topic_info().head(10)

,Topic,Count,Name,Representation,Representative_Docs
0,-1,2155,-1_drama_ficción_suspense_biografía,"[drama, ficción, suspense, biografía, comedia,...",[Diario de un escándalo. Una profesora de secu...
1,0,1869,0_asesino_asesinato_detective_crimen,"[asesino, asesinato, detective, crimen, drama,...",[Al sur de Los Ángeles. Un hombre recibe una c...
2,1,50,1_musical_cantante_música_músico,"[musical, cantante, música, músico, músicos, c...",[Stop Making Sense. Un innovador documental mu...
3,2,45,2_perro_dog_perros_mascota,"[perro, dog, perros, mascota, superdog, animal...","[Beethoven 2: La familia crece. Beethoven, el ..."
4,3,41,3_samurái_samurai_japoneses_japan,"[samurái, samurai, japoneses, japan, japón, ja...",[El ocaso del samurái. A medida que la era del...
5,4,40,4_béisbol_baseball_league_baseketball,"[béisbol, baseball, league, baseketball, jugad...",[Una pandilla de pelotas. El entrenador de la ...
6,5,38,5_christmas_santa_regalo_halloween,"[christmas, santa, regalo, halloween, seth, ju...",[Santa Claus: La película. La leyenda de Papá ...
7,6,34,6_piratas_pirata_pirate_caribe,"[piratas, pirata, pirate, caribe, tiburones, t...",[Piratas del Caribe: El cofre del hombre muert...
8,7,26,7_nazi_holocausto_nazis_judíos,"[nazi, holocausto, nazis, judíos, alemania, ge...",[La ladrona de libros. Cuando se ve sometido a...
9,8,21,8_caballo_caballos_horse_cowboy,"[caballo, caballos, horse, cowboy, ranchero, g...","[Babe: El cerdito en la ciudad. Babe, recién l..."


In [66]:
print("Documentos representativos del tópico 1:\n")
for doc in topic_model.get_representative_docs(1):
    print(doc)
    print("---")

Documentos representativos del tópico 1:

Stop Making Sense. Un innovador documental musical sobre un concierto del grupo de rock Talking Heads. Jonathan Demme. 1984. documental, música. película de concierto, live in concert recording, registro nacional de películas, canción, banda
---
Purple Rain. Una historia humana de supervivencia y triunfo, con la presencia de la estrella de rock Pince, quien interpreta a un joven de Minneapolis que lucha por ganar aceptación para su música rock. Albert Magnoli. 1984. drama, música, romance. skinny dipping, reference to lake minnetonka, cantante, la década de 1980, female frontal nudity
---
Rock Star. El cantante principal de una banda tributo se convierte en el cantante principal de la banda real que él idolatra. Stephen Herek. 2001. drama, música. tribute band, cover band, estrella de rock, heavy metal, banda de rock
---


### Representación del usuario en el espacio de tópicos

Para cada usuario concatenamos su **query + el texto de su historial** y usamos `topic_model.transform` para obtener su **distribución sobre los tópicos**. Así el usuario queda representado en el mismo espacio que las películas y podemos compararlos por similitud coseno entre distribuciones.

In [67]:
def build_user_text(usuario_row, pelis_df):
    """Construye el texto concatenado para un usuario"""
    user_text = usuario_row['query']
    
    for col in ['pelicula_1','pelicula_2','pelicula_3','pelicula_4','pelicula_5']:
        nombre = usuario_row[col]
        peli = pelis_df[pelis_df['name'] == nombre]
        if not peli.empty:
            user_text += " " + peli["texto"].values[0]
    
    return user_text

In [68]:
usuarios['texto'] = usuarios.apply(lambda row: build_user_text(row, df_pelis), axis=1)

In [69]:
_, users_probs = topic_model.transform(usuarios['texto'].tolist())

print(f"users_probs shape: {users_probs.shape}")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-18 23:30:56,209 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-06-18 23:30:58,073 - BERTopic - Dimensionality - Completed ✓
2026-06-18 23:30:58,074 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-06-18 23:30:58,078 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-06-18 23:30:58,108 - BERTopic - Probabilities - Completed ✓
2026-06-18 23:30:58,109 - BERTopic - Cluster - Completed ✓


users_probs shape: (14, 74)


### Recomendaciones

In [70]:
# Calcular similitud coseno
scores_tm = cosine_similarity(users_probs, probs)

# Crear máscara de películas ya vistas
scores_filtered = scores_tm.copy()

for i, row in usuarios.iterrows():
    historial_names = [row['pelicula_1'], row['pelicula_2'], row['pelicula_3'], 
                       row['pelicula_4'], row['pelicula_5']]
    
    # Encontrar índices de películas en el historial
    historial_indices = df_pelis[df_pelis['name'].isin(historial_names)].index
    
    # Asignar -inf para que no salgan en top-5
    scores_filtered[i, historial_indices] = -np.inf

# Top-5 por usuario
top5_indices_tm = scores_tm.argsort(axis=1)[:, -5:][:, ::-1]

# Ver resultados
for i, row in usuarios.iterrows():
    print(f"\n{row['nombre']} ({row['tipo_perfil']})")
    print(f"Query: {row['query']}")
    for idx in top5_indices_tm[i]:
        pelicula = df_pelis.iloc[idx]
        print(f"  {pelicula['name']} ({int(pelicula['year']) if pd.notnull(pelicula['year']) else 'N/A'}) — {scores_tm[i, idx]:.4f}")

ValueError: Incompatible dimension for X and Y matrices: X.shape[1] == 74 while Y.shape[1] == 73

### Validación

In [48]:
etiquetas_a_ojo_def = [
    ["suspense", "terror", "drama"],
    ["crimen", "biografía", "historia"],
    ["comedia", "romance", "drama"],
    ["acción", "ciencia ficción", "suspense"],
    ["animación", "drama", "aventura"],
    ["crimen", "acción", "comedia"],
    ["música", "drama", "comedia"],
    ["acción", "crimen", "aventura"],
    ["drama", "romance", "comedia"],
    # el U10 es ambiguo, la etiqueta debe estar mal
    # los demás usuarios son ambiguos
]

In [49]:
resultados_eval = []

for user_idx, row in usuarios.head(9).iterrows():
    print(f"\n{'='*70}")
    print(f"{row['nombre']} ({row['tipo_perfil']})")
    print(f"Query: {row['query']}")
    generos_esperados = set(etiquetas_a_ojo_def[user_idx])
    print(f"Géneros Esperados: {', '.join(generos_esperados)}")
    print(f"{'='*70}")
    
    generos_recomendados = Counter()
    peliculas_buenas = 0  # con al menos 1 género esperado
    peliculas_malas = []  # sin ningún género esperado
    
    print("\nTop-5 Recomendaciones:")
    for rank, idx in enumerate(top5_indices_tm[user_idx], 1):
        pelicula = df_pelis.iloc[idx]
        score = scores_tm[user_idx, idx]
        
        # Extraer géneros
        generos_str = pelicula['genre'].strip('[]')
        generos_list = [g.strip() for g in generos_str.split(',')]
        generos_pelicula = set(generos_list)
        generos_recomendados.update(generos_list)
        
        # ¿Tiene algún género esperado?
        es_buena = bool(generos_pelicula & generos_esperados)
        if es_buena:
            peliculas_buenas += 1
            marker = "✓"
        else:
            peliculas_malas.append(pelicula['name'])
            marker = "✗"
        
        print(f"  {rank}. [{marker}] {pelicula['name']} ({int(pelicula['year']) if not pd.isna(pelicula['year']) else 'N/A'}) — {score:.4f}")
        print(f"     Géneros: {', '.join(generos_list)}")
    
    # Conteo de géneros
    print(f"\nConteo de Géneros en Recomendaciones:")
    for genero, freq in generos_recomendados.most_common():
        print(f"  {genero}: {freq}")
    
    # Métricas
    generos_capturados = set(generos_recomendados.keys())
    recall = len(generos_capturados & generos_esperados) / len(generos_esperados)
    precision = peliculas_buenas / 5  # top-5
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    print(f"\nMÉTRICAS:")
    print(f"  Recall (géneros):     {recall:.1%}  ({len(generos_capturados & generos_esperados)}/{len(generos_esperados)})")
    print(f"  Precision (películas): {precision:.1%}  ({peliculas_buenas}/5)")
    print(f"  F1-Score:            {f1:.1%}")
    
    if peliculas_malas:
        print(f"\nPelículas problemáticas (sin géneros esperados):")
        for pelicula in peliculas_malas:
            print(f"    - {pelicula}")
    
    resultados_eval.append({
        'Usuario': row['nombre'],
        'Recall': recall,
        'Precision': precision,
        'F1': f1,
        'Películas Malas': len(peliculas_malas)
    })

# Resumen
print(f"\n\n{'='*70}")
print("RESUMEN DE EVALUACIÓN")
print(f"{'='*70}")
df_eval = pd.DataFrame(resultados_eval)
df_eval.to_csv("evaluacion_topic_modeling.csv", index=False)
print(df_eval.to_string(index=False))
print(f"\nPromedios:")
print(f"  Recall:    {df_eval['Recall'].mean():.1%}")
print(f"  Precision: {df_eval['Precision'].mean():.1%}")
print(f"  F1-Score:  {df_eval['F1'].mean():.1%}")


Valentina (definido)
Query: Quiero una película donde una mujer enfrenta una amenaza invisible que viene de alguien cercano
Géneros Esperados: drama, terror, suspense

Top-5 Recomendaciones:
  1. [✗] El padre de la novia (1992) — 0.9999
     Géneros: comedia, familiar, romance
  2. [✓] La boda de mi mejor amigo (1997) — 0.9999
     Géneros: comedia, drama, romance
  3. [✗] Serendipity (2002) — 0.9999
     Géneros: comedia, romance
  4. [✓] Pastoral americana (2017) — 0.9999
     Géneros: crimen, drama
  5. [✓] Las brujas de Eastwick (1987) — 0.9999
     Géneros: comedia, fantasía, terror

Conteo de Géneros en Recomendaciones:
  comedia: 4
  romance: 3
  drama: 2
  familiar: 1
  crimen: 1
  fantasía: 1
  terror: 1

MÉTRICAS:
  Recall (géneros):     66.7%  (2/3)
  Precision (películas): 60.0%  (3/5)
  F1-Score:            63.2%

Películas problemáticas (sin géneros esperados):
    - El padre de la novia
    - Serendipity

Rodrigo (definido)
Query: Busco algo basado en hechos reales sobr

Este enfoque parece no ser el ideal para el problema. Algunos usuarios (Camila, Martín) alcanzaron F1=100%, otros (Lucía) F1=0%, sugiriendo que el método no generaliza bien.